In [1]:
import json
import re
import csv

# ── Load the JSON ─────────────────────────────────────────────────────────────
with open(r'C:\Users\nasee\Downloads\Gym.json', encoding='utf-8') as f:
    data = json.load(f)

# ── Fix date typos ────────────────────────────────────────────────────────────
data['textContent'] = data['textContent'].replace('2/9/26', '2/9/25')
data['textContent'] = data['textContent'].replace('6/9/26', '6/9/25')
data['textContent'] = data['textContent'].replace('9/9/26', '9/9/25')
data['textContent'] = data['textContent'].replace('13/9/26', '13/9/25')

# ── Exercise name aliases ─────────────────────────────────────────────────────
ALIASES = {
    "Inclined Smith Bench Press": "Inclined Smith Bench Press",
    "Inclined Smith Bench": "Inclined Smith Bench Press",
    "Inclined Smith Machine Bench Press": "Inclined Smith Bench Press",
    "Inclined Smith Machine Bench": "Inclined Smith Bench Press",
    "Inclined Smith Machine Press": "Inclined Smith Bench Press",
    "Inclined Smith Machine": "Inclined Smith Bench Press",
    "Inclined Smich Bench Press": "Inclined Smith Bench Press",
    "Inclined Smich Bench (Steep)": "Inclined Smith Bench Press",
    "Inclined Smitch Bench Press": "Inclined Smith Bench Press",
    "Inclined Smich Machine Bench (Old Mech)": "Inclined Smith Bench Press",
    "Inclined Smith Machine Benxh": "Inclined Smith Bench Press",
    "Incline Smith Bench": "Inclined Smith Bench Press",
    "Smith Bench Press": "Inclined Smith Bench Press",
    "Smith Bench": "Inclined Smith Bench Press",
    "Smith Inclined Bench Press": "Inclined Smith Bench Press",
    "Smith Inclined Bench": "Inclined Smith Bench Press",
    "Smith Inclined Chest": "Inclined Smith Bench Press",
    "Smith Machine Bench Press": "Inclined Smith Bench Press",
    "Easy Smitch Inclined Bench": "Inclined Smith Bench Press",
    "Inclined Smith Bench Press (Easy)": "Inclined Smith Bench Press",
    "Smich Inclined Bench Press": "Inclined Smith Bench Press",
    "Smitch Machine Inclined Bench Press": "Inclined Smith Bench Press",
    "Incined Smith Bench Press": "Inclined Smith Bench Press",
    "Inclined Bench Press Machine": "Inclined Machine Bench Press",
    "Inclined Chest Press Machine": "Inclined Machine Bench Press",
    "Inclined Machine Bench Press": "Inclined Machine Bench Press",
    "Inclined Machine Chest Press": "Inclined Machine Bench Press",
    "Inclined Machine Press": "Inclined Machine Bench Press",
    "Machine Inclined Bench": "Inclined Machine Bench Press",
    "Machine Inclined Bench Press": "Inclined Machine Bench Press",
    "Machine Inclined Chest": "Inclined Machine Bench Press",
    "Machine Inclined Chest Press": "Inclined Machine Bench Press",
    "Machine Inclined Press": "Inclined Machine Bench Press",
    "Machine Bench Press": "Inclined Machine Bench Press",
    "Inclined Bench Press": "Inclined Machine Bench Press",
    "Inclined Dumbell Bench": "Inclined Dumbbell Bench Press",
    "Lat Pull Down": "Lat Pull Down",
    "Lat Pull Down (Crap Machine)": "Lat Pull Down",
    "Lat Pull Down (Wide Grip)": "Lat Pull Down",
    "Lat Pushdown": "Lat Pull Down",
    "Preacher Curl": "Preacher Curl",
    "Machine Preacher Curl": "Preacher Curl",
    "Machine Preacher Curl :": "Preacher Curl",
    "Preacher Curl Machine": "Preacher Curl",
    "Preacher Machine Curl": "Preacher Curl",
    "Preacher": "Preacher Curl",
    "Back Row": "Back Row",
    "Backrow": "Back Row",
    "Back Cable Pull": "Back Row",
    "Cable Back Row": "Back Row",
    "Cable Row": "Back Row",
    "Machine Back Row": "Back Row",
    "Side Raise": "Side Delt Raise",
    "Side Shoulder Raise": "Side Delt Raise",
    "Side Shoulder Cable": "Side Delt Raise",
    "Cable Side Raise": "Side Delt Raise",
    "Shoulder Press": "Shoulder Press",
    "Machine Shoulder Press": "Shoulder Press",
    "Tricep Push Down": "Tricep Push Down",
    "Tricep Pushdown": "Tricep Push Down",
    "Chest Fly": "Machine Chest Fly",
    "Chest Fly (Max Resistance)": "Machine Chest Fly",
    "Machine Chest Fly": "Machine Chest Fly",
    "Pectoral Machine": "Machine Chest Fly",
    "Smith Squats": "Smith Squats",
    "Smuth Squat": "Smith Squats",
    "Reverse Fly Rear Delt": "Rear Delt Fly",
}

# ── Muscle groups ─────────────────────────────────────────────────────────────
MUSCLE_GROUPS = {
    "Inclined Smith Bench Press":       "Chest",
    "Inclined Machine Bench Press":     "Chest",
    "Inclined Dumbbell Bench Press":    "Chest",
    "Machine Chest Fly":                "Chest",
    "Lat Pull Down":                    "Back",
    "Back Row":                        "Back",
    "Preacher Curl":                    "Biceps",
    "Tricep Push Down":                 "Triceps",
    "Tricep Extension":                 "Triceps",
    "Shoulder Press":                   "Shoulders",
    "Side Delt Raise":                  "Shoulders",
    "Rear Delt Fly":                    "Shoulders",
    "Leg Press":                        "Legs",
    "Leg Curl":                         "Legs",
    "Leg Extension":                    "Legs",
    "Smith Squats":                     "Legs",
    "Rdl":                              "Legs",
}

def normalise_exercise(name):
    return ALIASES.get(name, name)

def get_muscle_group(exercise):
    return MUSCLE_GROUPS.get(exercise, "Other")

# ── Parse the data ────────────────────────────────────────────────────────────
text = data['textContent']
lines = text.split('\n')

date_re = re.compile(r'^\d{1,2}/\d{1,2}/\d{2,4}$')
set_re  = re.compile(r'(\d+\.?\d*)\s*kg\s*[-–]\s*(\d+)', re.IGNORECASE)

rows = []
current_date     = None
current_exercise = None
set_number       = 0

for line in lines:
    line = line.strip()
    if not line:
        continue

    if date_re.match(line):
        d, m, y = line.split('/')
        if len(y) == 2:
            y = '20' + y
        current_date     = f"{y}-{m.zfill(2)}-{d.zfill(2)}"
        current_exercise = None
        set_number       = 0

    elif set_re.search(line):
        match    = set_re.search(line)
        weight   = float(match.group(1))
        reps     = int(match.group(2))
        set_number += 1
        exercise = normalise_exercise(current_exercise or 'Unknown')
        rows.append({
            'date':         current_date,
            'exercise':     exercise,
            'muscle_group': get_muscle_group(exercise),
            'set_number':   set_number,
            'weight_kg':    weight,
            'reps':         reps,
            'volume_kg':    round(weight * reps, 2),
        })

    else:
        current_exercise = line.title().strip()
        set_number       = 0

# ── Save to CSV ───────────────────────────────────────────────────────────────
out_path = r'C:\Users\nasee\OneDrive\Documents\Data Portfolio\Power BI Portfolio\Gym Tracker\gym_data.csv'
fields   = ['date', 'exercise', 'muscle_group', 'set_number', 'weight_kg', 'reps', 'volume_kg']

with open(out_path, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=fields)
    writer.writeheader()
    writer.writerows(rows)

# ── Summary ───────────────────────────────────────────────────────────────────
exercises = sorted(set(r['exercise'] for r in rows))
dates     = sorted(set(r['date'] for r in rows))

print(f"✅  {len(rows)} sets across {len(dates)} sessions")
print(f"✅  {len(exercises)} unique exercises")
print(f"✅  Saved to {out_path}")
print(f"\nDate range: {dates[0]}  →  {dates[-1]}")

✅  977 sets across 63 sessions
✅  17 unique exercises
✅  Saved to C:\Users\nasee\OneDrive\Documents\Data Portfolio\Power BI Portfolio\Gym Tracker\gym_data.csv

Date range: 2025-09-02  →  2026-05-09


In [2]:
import pandas as pd

df = pd.read_csv(r'C:\Users\nasee\OneDrive\Documents\Data Portfolio\Power BI Portfolio\Gym Tracker\gym_data.csv')
df

,date,exercise,muscle_group,set_number,weight_kg,reps,volume_kg
0,2026-05-09,Inclined Smith Bench Press,Chest,1,40.0,8,320.0
1,2026-05-09,Inclined Smith Bench Press,Chest,2,60.0,6,360.0
2,2026-05-09,Inclined Smith Bench Press,Chest,3,60.0,5,300.0
3,2026-05-09,Inclined Smith Bench Press,Chest,4,60.0,3,180.0
4,2026-05-09,Lat Pull Down,Back,1,70.0,12,840.0
...,...,...,...,...,...,...,...
972,2025-10-25,Machine Chest Fly,Chest,3,68.0,6,408.0
973,2025-10-25,Tricep Push Down,Triceps,1,30.0,8,240.0
974,2025-10-25,Tricep Push Down,Triceps,2,35.0,4,140.0
975,2025-10-25,Tricep Push Down,Triceps,3,32.5,5,162.5
